# TRAG – model procesa za R3 recikliranje plastike

ROO 2023. Za svaku lokaciju kojoj je R3 recikliranje plastike glavna djelatnost računamo **očekivani raspon prijavljenog ostatka** iz sastava ulaza i uspoređujemo ga s prijavljenim ostatkom. Time omjer iz prvog dema postaje test.

Uz notebook trebaju biti `obrada_2023.xlsx` i `nastanak_2023.xlsx`. Pokreni ćelije redom.

**Pristup je namjerno konzervativan:** u prijavljeni ostatak ulazi sav otpad nastao na lokaciji, a ulazu koji nije plastika dajemo donju granicu 0 %. Obje pretpostavke smanjuju broj označenih lokacija. Rezultat je odstupanje za provjeru, a ne dokaz.

## 1. Parametri

Sve pretpostavke su ovdje. Rasponi su udio **prijavljenog** ostatka u ulazu (vlaga i prljavština koje ispare ili odu u kanalizaciju nisu prijavljeni otpad).

| Razred | Ulaz | Osnovni raspon | Blagi raspon |
|---|---|---|---|
| A | čisti industrijski otpad (02 01 04, 07 02 13, 12 01 05) | 1–5 % | 0,5–5 % |
| B | sortirani potrošački otpad (15 01 02, 16 01 19, 17 02 03, 20 01 39) | 10–35 % | 5–35 % |
| C | sekundarni otpad iz mehaničke obrade (19 12 04, 19 12 12) | 30–60 % | 10–60 % |

**Izvor i nesigurnost.** Za sortiranu ambalažu JRC je izmjerio iskorištenje recikližnih pogona 63–90 % za PET, 70–88 % za HDPE i 53–85 % za PP (Antonopoulos, Faraca i Tonini, *Waste Management* 2021, doi:10.1016/j.wasman.2021.04.002), dakle ukupni gubitak 10–47 %. Rasponi za A i C, kao i podjela na prijavljeni i neprijavljeni gubitak, **procjena su tima** i treba ih potvrditi literaturom i mjerenjima. Blagi scenarij postoji upravo zato: ako lokacija odstupa i u njemu, zaključak ne ovisi o točnim granicama.

Razred C je najveća nesigurnost. 19 12 04 može biti i sortirana frakcija iz sortirnice, koja je čišća od miješanog ostatka. Blagi scenarij zato spušta donju granicu C na razinu sortiranog otpada (B).

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

RAZREDI = {
    "A": ["02 01 04", "07 02 13", "12 01 05"],
    "B": ["15 01 02", "16 01 19", "17 02 03", "20 01 39"],
    "C": ["19 12 04", "19 12 12"],
}
# (donja, gornja) granica udjela prijavljenog ostatka
SCENARIJI = {
    "osnovni": {"A": (0.01, 0.05), "B": (0.10, 0.35), "C": (0.30, 0.60), "ostalo": (0.0, 1.0)},
    "blagi":   {"A": (0.005, 0.05), "B": (0.05, 0.35), "C": (0.10, 0.60), "ostalo": (0.0, 1.0)},
}

# Odabir lokacija: plastika pod R3 je barem 80 % stvarne obrade i barem 500 t.
MIN_UDIO_PLASTIKE = 0.80
MIN_PLASTIKA_T = 500
# Postupci koji nisu stvarna obrada (skladištenje, priprema, prethodni postupci).
NIJE_OBRADA = ["R12", "R13", "D15", "PP"]

# Kapacitetni test
RADNIH_SATI = 250 * 16          # 250 dana, dvije smjene
LINIJA_T_H = (0.5, 2.0)         # tipični učinak jedne linije pranja i granuliranja, t/h (procjena)
ENERGIJA_KWH_KG = (0.5, 1.0)    # potrošnja cijele linije, kWh/kg (procjena)
# Kapacitet iz dozvole po OIB-u, t/god. Upiši kad dobiješ rješenja o dozvoli.
KAPACITET_DOZVOLA = {}

## 2. Učitavanje podataka

Lokacija je određena OIB-om i HTRS koordinatama, što je otpornije od naziva i adrese.

In [ ]:
obrada = pd.read_excel("obrada_2023.xlsx", dtype={"OIB": str})
nastanak = pd.read_excel("nastanak_2023.xlsx", dtype={"OIB": str})

nazivi = {
    "OIB": "oib",
    "HTRS 96 TM koordinate organizacijske jedinice (E)": "e",
    "HTRS 96 TM koordinate organizacijske jedinice (N)": "n",
    "Operater": "operater",
    "Grad/naselje organizacijske jedinice": "mjesto",
    "Ključni broj otpada": "otpad",
    "Postupanje s otpadom - postupak": "postupak",
    "Postupanje s otpadom - količina (t)": "t",
    "Nastalo (t)": "nastalo",
}
obrada = obrada.rename(columns=nazivi)
nastanak = nastanak.rename(columns=nazivi)
LOK = ["oib", "e", "n"]

plasticni = [k for lista in RAZREDI.values() for k in lista]
razred_od = {k: r for r, lista in RAZREDI.items() for k in lista}

stvarna = obrada[~obrada["postupak"].isin(NIJE_OBRADA)].copy()
stvarna["razred"] = "ostalo"
r3_plastika = (stvarna["postupak"] == "R3") & stvarna["otpad"].isin(plasticni)
stvarna.loc[r3_plastika, "razred"] = stvarna.loc[r3_plastika, "otpad"].map(razred_od)

sastav = stvarna.pivot_table(index=LOK, columns="razred", values="t", aggfunc="sum", fill_value=0)
for r in ["A", "B", "C", "ostalo"]:
    if r not in sastav:
        sastav[r] = 0.0
sastav["ulaz_t"] = sastav[["A", "B", "C", "ostalo"]].sum(axis=1)
sastav["plastika_t"] = sastav[["A", "B", "C"]].sum(axis=1)

odabir = sastav[(sastav["plastika_t"] >= MIN_PLASTIKA_T)
                & (sastav["plastika_t"] / sastav["ulaz_t"] >= MIN_UDIO_PLASTIKE)].copy()

ostatak = nastanak.groupby(LOK)["nastalo"].sum()
odabir = odabir.join(ostatak.rename("ostatak_t"), how="left")
odabir = odabir.join(obrada.groupby(LOK)[["operater", "mjesto"]].first())
odabir["mjesto"] = odabir["mjesto"].fillna("mjesto nije upisano")

if odabir["ostatak_t"].isna().any():
    print("Upozorenje: lokacije bez prijave nastanka otpada:")
    print(odabir[odabir["ostatak_t"].isna()][["operater", "mjesto"]])
odabir["ostatak_t"] = odabir["ostatak_t"].fillna(0)
print(f"Odabrano lokacija: {len(odabir)}")

## 3. Očekivani raspon i test

Očekivani raspon lokacije je ponderirani prosjek raspona razreda prema sastavu ulaza. **Faktor odstupanja** je donja granica podijeljena s prijavljenim omjerom: 1 znači točno na granici, 10 znači deset puta manje ostatka od najmanjeg očekivanog.

Oznake (dogovor tima, ne standard): faktor do 2 je unutar nesigurnosti modela, 2–10 je odstupanje, iznad 10 je jako odstupanje.

In [ ]:
def ocjena(faktor):
    if faktor <= 1:
        return "u rasponu"
    if faktor <= 2:
        return "unutar nesigurnosti"
    if faktor <= 10:
        return "odstupanje"
    return "jako odstupanje"

udjeli = odabir[["A", "B", "C", "ostalo"]].div(odabir["ulaz_t"], axis=0)
odabir["omjer"] = odabir["ostatak_t"] / odabir["ulaz_t"]

for ime, rasponi in SCENARIJI.items():
    donja = sum(udjeli[r] * rasponi[r][0] for r in rasponi)
    gornja = sum(udjeli[r] * rasponi[r][1] for r in rasponi)
    odabir[f"donja_{ime}"] = donja
    odabir[f"gornja_{ime}"] = gornja
    odabir[f"faktor_{ime}"] = donja / odabir["omjer"].clip(lower=1e-6)
    odabir[f"ocjena_{ime}"] = odabir[f"faktor_{ime}"].map(ocjena)

odabir["glavni_razred"] = udjeli[["A", "B", "C"]].idxmax(axis=1)
odabir = odabir.sort_values("faktor_osnovni", ascending=False)

tablica = odabir.reset_index()[[
    "operater", "mjesto", "glavni_razred", "ulaz_t", "ostatak_t", "omjer",
    "donja_osnovni", "faktor_osnovni", "ocjena_osnovni", "donja_blagi", "faktor_blagi", "ocjena_blagi",
]].copy()
for s in ["omjer", "donja_osnovni", "donja_blagi"]:
    tablica[s] = (100 * tablica[s]).round(2)
tablica["operater"] = tablica["operater"].str.slice(0, 32)
tablica = tablica.rename(columns={
    "operater": "Operater", "mjesto": "Mjesto", "glavni_razred": "Razred",
    "ulaz_t": "Ulaz (t)", "ostatak_t": "Ostatak (t)", "omjer": "Ostatak (%)",
    "donja_osnovni": "Min. osnovni (%)", "faktor_osnovni": "Faktor osn.", "ocjena_osnovni": "Ocjena osn.",
    "donja_blagi": "Min. blagi (%)", "faktor_blagi": "Faktor blagi", "ocjena_blagi": "Ocjena blagi",
})
display(tablica.round({"Ulaz (t)": 0, "Ostatak (t)": 1, "Faktor osn.": 1, "Faktor blagi": 1}))

In [ ]:
fig, ax = plt.subplots(figsize=(9, 0.55 * len(odabir) + 1.6), layout="constrained")
redovi = odabir.reset_index()
y = range(len(redovi))

for i, r in redovi.iterrows():
    ax.plot([100 * r["donja_blagi"], 100 * r["gornja_blagi"]], [i, i], color="#d4dde3", lw=9, solid_capstyle="butt")
    ax.plot([100 * r["donja_osnovni"], 100 * r["gornja_osnovni"]], [i, i], color="#a7b8c3", lw=9, solid_capstyle="butt")
    boja = "#b8442f" if r["faktor_blagi"] > 2 else "#23657e"
    ax.scatter(max(100 * r["omjer"], 0.01), i, color=boja, s=70, zorder=3)

ax.set_yticks(list(y), labels=[f"{o[:28]} ({m})" for o, m in zip(redovi["operater"], redovi["mjesto"])])
ax.invert_yaxis()
ax.set_xscale("log")
ax.set_xticks([0.1, 1, 10, 100], labels=["0,1", "1", "10", "100"])
ax.set_xlabel("Prijavljeni ostatak / ulaz (%) — log. skala")
ax.set_title("Prijavljeni ostatak (točka) i očekivani raspon
(tamna traka: osnovni scenarij, svijetla: blagi)", loc="left", fontsize=10)
ax.grid(axis="x", alpha=0.2)
ax.set_axisbelow(True)
plt.show()

## 4. Kapacitetni test

Koliki bi učinak i potrošnja energije bili potrebni da je prijavljena količina stvarno obrađena. Bez podataka iz dozvole ovo je samo orijentacija: velik pogon s velikim ulazom je normalan. Test postaje oštar kad se u `KAPACITET_DOZVOLA` upiše dopušteni kapacitet, a iz dozvole ili elaborata broj i učinak linija.

In [ ]:
kap = odabir.reset_index()[["oib", "operater", "mjesto", "plastika_t"]].copy()
kap["ucinak_t_h"] = kap["plastika_t"] / RADNIH_SATI
kap["linija_min"] = (kap["ucinak_t_h"] / LINIJA_T_H[1]).round(1)
kap["linija_max"] = (kap["ucinak_t_h"] / LINIJA_T_H[0]).round(1)
kap["energija_GWh_min"] = kap["plastika_t"] * ENERGIJA_KWH_KG[0] / 1000
kap["energija_GWh_max"] = kap["plastika_t"] * ENERGIJA_KWH_KG[1] / 1000
kap["kapacitet_dozvola_t"] = kap["oib"].map(KAPACITET_DOZVOLA)
kap["iznad_dozvole"] = (kap["plastika_t"] > kap["kapacitet_dozvola_t"]).where(kap["kapacitet_dozvola_t"].notna())
kap["operater"] = kap["operater"].str.slice(0, 32)
display(kap.drop(columns="oib").round(2))

## 5. Što rezultat znači

Tumačenje je napisano za ROO 2023. i ovdje navedene raspone. Ako promijeniš parametre, provjeri vrijedi li i dalje.

- Gospić odstupa u oba scenarija i za dva reda veličine. Zaključak ne ovisi o točnim granicama: prijavljeni ostatak je ispod i najblaže granice za najčišći razred ulaza (A), iako je ulaz sekundarni otpad (C).
- Gvozd odstupa u oba scenarija, ali za red veličine manje. Ulaz mu je također gotovo sav 19 12 04. Prije bilo kakvog zaključka treba provjeriti što je stvarno taj ulaz (sortirana frakcija ili miješani ostatak) i što je u dozvoli.
- Omjer iznad 100 % (Donja Dubrava, Ludbreg) znači da je na lokaciji prijavljeno više nastalog otpada nego obrade, vjerojatno iz druge proizvodnje na istoj lokaciji ili iz zaliha. Model provjerava samo donju granicu, pa ih ne označava.
- Lokacije s faktorom do 2 su unutar nesigurnosti modela. Model ih ne može razlikovati od normalnog rada dok se rasponi ne suze.
- Kapacitetni test zasad samo pokazuje potreban broj linija. Za Gospić to je 2 do 8 linija i 8–16 GWh struje godišnje; to treba usporediti s opremom i priključnom snagom iz dozvole.

**Sljedeći koraci:** potvrditi raspone razreda iz literature (posebno C), dobiti rješenja o dozvolama za odabrane lokacije, ponoviti za 2019.–2025.